# Bias, Variance, and the Trade-off



<img src="images/lec06_bias_var.png" width="500">

**Disclaimer**: The word *Bias* here relates to a mathematical concept, and not to the *human bias* - a judgement based on a person's view of the world. Same word but two different meanings.

## How would you draw a line through this?

Suppose we cold-work a metal - rolling or drawing it - and measure its **yield strength** at different amounts of cold work. The metal work-hardens: strength increases, then flattens out. We all know what a yield curve looks like right? The *true* relationship is a smooth curve. But when we are measuring it, all we ever see is a scatter of dots.

Something like this..

<img src="images/lec06_plot.png" width="500">


Now, the question is how do you draw a line through it? 

If you know the true relationship between the variables then that's easy, but in real-life we use machine learning when we **don't know** what the true relationship is!


## Let's look at the options

We'll fit the same data three ways: a **straight line** (degree 1), a **gentle curve**
(degree 3), and a **wildly flexible** polynomial (degree 15). And watch what each one does.

<img src="images/lec06_3_plots.png" width="800">

Three very different stories:

- **Degree 1 (green)** - a straight line. It's *stubborn*: no matter how the dots
  curve, it insists on being straight. It misses the plateau (flattened section beyond ~ 60%) completely. This is
  **high bias**.
- **Degree 15 (red)** - it bends to touch nearly every dot, wiggling through the noise.
  It looks brilliant *here*, and if you calculate the *sum of squares error* this will have the lowest value, but it's trying to fit exactly to each data point - This is **high variance**. We can also say that it is *memorizing* the data.
- **Degree 3 (purple)** - Is more flexible than the first one, but not as much as the degree 25 - somewhere in the middle. It seems to follow the real shape without chasing every dot. **Just right.**

:::{important} Definitions, in plain words
- **Bias** is the error you get from *over-simplifying*. A high-bias model is like someone
  who answers every question with "it's probably linear" - consistent, but also consistently off.
- **Variance** is how much your model *changes its mind* when the data changes a little.
  A high-variance model is like someone who completely rewrites their theory every time
  one new data point walks in.

Note that "high bias" here is *relative to the true relationship*. A straight line isn't
biased by nature - it's biased **because the real pattern is curved** and a line can't bend.
:::


Try playing around with the parameters in this **⏯️ [interactive visualization](data/polynomial_playground.html)** and see what happens.


## Why not just have both low?

Fair question. Why not pick a model with *zero* bias **and** *zero* variance?

⚠️ Because they pull in opposite directions. Just like two kids on a see-saw: when one goes up,the other one goes down. 
Make a model more flexible and it fits the training data better (bias ↓) - but it also starts chasing noise (variance ↑). 
Make itsimpler and it stops chasing noise (variance ↓) - but now it can't follow the real shape(bias ↑). 
That tug-of-war *is* the bias–variance trade-off.

A good way to see it: fit polynomials of increasing degree, and track the error on
the **training** set versus a held-out **test** set.

You can try doing this with the above interactive visualization. Once you record the numbers you should see something like below.

<img src="images/lec06_train_test_tradeoff.png" width="800">

Read this plot slowly - it's the whole chapter in one picture:

- **Training error (green)** just keeps dropping. Of course it does: give a model more
  flexibility and it can always hug the training dots tighter. *Training error alone
  will keep on decreasing - even if the decrease is very small, it doesn't tell you when to stop.*
- **Test error (red)** is **U-shaped**. It falls, bottoms out, then climbs again. The
  bottom of that U is the sweet spot - here, **degree 3**.
  Why?? 🤔 - you should be able to explain why this happens.\
  
- To the **left** of the U: too simple, high bias, *underfitting*.
- To the **right**: too flexible, high variance, *overfitting*.

:::{warning} The overfit 😖

The degree-15 model gets a *fantastic* training RMSE of about **9 MPa** - it practically
memorised the training set. But on new test data its RMSE explodes to **~530 MPa** and its
$R^2$ goes **negative** ($\approx -28$). A negative $R^2$ means it's doing *worse than just
predicting the average every time*. It aced the practice exam and then failed the real one
spectacularly. 
:::


## How do we measure "good"? The metrics

👀 Eyeballing plots is great for intuition, but we need numbers to actually prove something. 

We will talk about the following four metrics:

$R^2$, **MAE**, **RMSE**, and **residual analysis**. Let's put real numbers on our three fits.


<img src="images/lec06_metric_table.png" width="600">

Look down the columns:

- **Degree 1** is mediocre everywhere - this is a sign of **high bias**. 🤷‍♂️
- **Degree 3** is good on training *and* test, and the two are close. That closeness is
  exactly what we want: it means the model **generalises**.👍
- **Degree 15** looks amazing on training and dreadful on test. That *gap* between train
  and test is the fingerprint of **high variance**.😨

:::{tip} The single most useful habit 💡 

Always compare **training vs test** performance. A model that's great on training but poor
on test is overfitting. If it's poor on both, it's underfitting. The *gap* tells you which
problem you have.
:::


### MAE vs RMSE - why two error metrics?

Both measure "how far off are we, on average", in the **same units as the target** (here, MPa).

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}\left|y_i - \hat{y}_i\right|
\qquad\qquad
\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat{y}_i\right)^2}$$

The difference is how they treat **big misses**:

- **MAE** treats every error fairly - a 10 MPa miss counts as 10.
- **RMSE** *squares* errors first, so a 10 MPa miss counts as 100 before averaging.
  Big mistakes get punished disproportionately, which makes **RMSE sensitive to outliers**.

👉 Rule of thumb: if one big mistake is much worse than several small ones (say, a strength
prediction that's off by 300 MPa, rather than several samples where the strength variation is ~10 MPa), then go for the **RMSE**. 

If all errors are equally annoying and although you have some wild outliers you don't want them dominating the score, **MAE**
is the calmer choice. 

And $R^2$ answers a different question - *what fraction of the variation did we explain?* - where **1.0 is perfect** and **0 is no better than the mean**.


### Residual analysis - the model's leftovers

A **residual** is what's left over after the model has done its job fitting to the data:

$$\text{residual} = y_i - \hat{y}_i \quad(\text{actual} - \text{predicted})$$

Here's the trick: if a model has truly captured the pattern, its residuals should look like
**random noise** - no shape, no trend. But if the residuals form a **pattern**, the model
has *missed something systematic*. Let's compare our underfit line and our good fit.

<img src="images/lec06_residuals.png" width="700">

On the **left**, the residuals sweep from negative to positive and back to negative - a clear **U** (upside.
The straight line is systematically too high (compared to data points) in the middle and too low at the ends, because
it can't bend to match the data pattern. The residual plot is missing curvature!

On the **right**, the residuals scatter randomly around zero with no obvious shape. That's the
look of a model that has nothing systematic left to explain - the leftovers are just noise.

:::{note} Why bother, when we already have $R^2$ and RMSE?
A single number is an overal value, and doesn't show *where* a model fails. 
Two models can share the same RMSE while one is uniformly okay and the other is great in one region and terrible in another. 
The residual plot shows you the *shape* of the mistake - often pointing straight at the fix (here: "use a curve").
:::


You can go back to the **⏯️ [interactive visualization](data/polynomial_playground.html)** and see how the residuals pattern change when you change the polynomial order. Everytime the bias is high (compared to the actual order of the data) the residuals show a pattern. 


## The same idea, everywhere 💭
## K-means

The bias-variance trade-off isn't just a regression thing. Take **K-means clustering**: Choose too few clusters and you combine different groups of materials together - an oversimplified, "high-bias" description. Choose too many clusters and each cluster starts capturing noise and quirks of *this particular* sample - a "high-variance" description that won't hold up on new data.

The **elbow method** you already know is really just looking for that balance point.


<img src="images/lec06_kmeans_bias_variance.png" width="700">

:::{caution} A technicality about K-means
Calling this a "bias–variance trade-off" is a useful **analogy**, not a literal one. The
formal bias–variance decomposition is defined for **supervised** prediction error (it splits
expected error into bias² + variance + irreducible noise). K-means has no labels and no
prediction error to decompose, so there's nothing to split in that exact sense.

What *is* real and shared is the **model-complexity story**: if it's too simple then it misses structure,if it's too complex then it chases noise, and there's a balance in between. The elbow is that balance for K-means. 

So: same intuition, same picture, different underlying maths.🙂
:::


## The knobs you turn: 🎛️ hyperparameters

Now, a model like linear regression is generally simple, but there are models such as Random Forests or Support Vector Machines (that we will meet soon!), which have additional parameters you can customize to fit your machine learning problem. So how do you know which parameters need adjusting, and to what level?

You need to be able to *dial in* the right amount of complexity through **hyperparameters** -
settings you choose **before** training, which the model does *not* learn on its own.

:::{admonition} The baking analogy 🍰
:class: tip
Think of your **data as the cake ingredients** and the **model as the oven**. The oven has
knobs: temperature, fan on/off, timer, top/bottom heat, rack level. Nobody hands you the
perfect combination - you try a few, taste the result, and adjust. Tuning a model's
hyperparameters is exactly that: try combinations, check the result on data the model hasn't
seen, keep what works. 
:::

Almost every model has these knobs:

| Model | A knob that controls bias ↔ variance |
|---|---|
| Ridge / Lasso regression | regularisation strength (how hard to shrink coefficients) |
| Logistic regression | `C` (inverse regularisation strength) |
| Decision trees | `max_depth`, `min_samples_leaf` |
| Random forests | number & depth of trees , etc.|
| SVM / SVC | `C` and the kernel's `gamma` |
| K-nearest neighbours | number of neighbours `k` |

### Example: `C` in logistic regression

In scikit-learn's `LogisticRegression`, the key knob is **`C`**, and it's a bit confusing
because it works *backwards* from what you'd guess:

- **Small `C`** → **strong** regularisation → simpler boundary → **higher bias**.
- **Large `C`** → **weak** regularisation → wigglier boundary → **higher variance**.

So `C` is the *inverse* of regularisation strength. Turn `C` down to calm an overfitting
model; turn it up if the model is too rigid. 

**Regularization** is basically telling your model to keep it simple 👮.

If the model tries too hard, it can happily memorize every bit of noise in your training data to nail it perfectly! then fall on its face when it sees new data because it learned the noise, not the underlying pattern. 

In Regularization you are acting like a policeman, and impose a little penalty for the model getting too complicated or too confident, nudging it toward simpler, more general solutions instead of overfit ones.


## So how do we actually find the balance?

You now know *what* the trade-off is and *how to spot it*. Next we will look at a toolkit for *managing* it:

- **Cross-validation** - a fair way to estimate test performance without wasting data.
  *(That's the very next part.)*
- **Balancing model complexity** - pick the flexibility that matches the problem, like
  choosing degree 3 instead of 1 or 15.
- **Regularisation** - gently penalise complexity so the model can't overfit (Ridge
  and Lasso, coming soon).
- **Hyperparameter tuning** - search the knob settings systematically instead of by hand (Grid Search,Randomized Search)
- **Ensemble methods** - combine many models (Random Forests,bagging, boosting) to cut variance or bias.

:::{admonition} One-line summary
:class: important 🎯
**Underfitting = high bias** (too simple, misses the pattern). **Overfitting = high variance**
(too flexible, memorises the noise). The art of machine learning is finding the *just-right*
middle - and **cross-validation** is how we find it fairly. ➡️
:::


## Finding good hyperparameters: grid and random search 🎛️ 

Hyperparameters like the penalty strength in Ridge and Lasso, the degree of a polynomial, or (soon) a kernel's length-scale all decide **where a model sits on the bias–variance line**. So how do we find good values? Two simple strategies come first.

**Grid search** lays down a regular grid of values and tries **every** combination. Simple and thorough - but the number of combinations explodes as you add more hyperparameters, and many trials land in poor regions.

**Random search** picks combinations **at random**. With the same number of trials it often does better - especially when only one or two hyperparameters really matter - because it samples more *distinct* values of each.


<img src="images/lec06_grid_random_search.png" width="700">

:alt: Grid search versus random search over a 2-D hyperparameter space
:width: 100%

Grid search (left) tests a regular lattice. Random search (right) scatters the **same number** of trials, so it tries more distinct values of each hyperparameter (the marks below each panel). Shaded = better settings; ★ = best found.
```

:::{note}
Both methods are **uninformed**: they never use the results already collected to decide where to look next. Every trial is blind to the last.
:::

That blindness is exactly the gap the next method closes: **Bayesian Optimization** uses what it has learned so far to choose each new trial wisely. We will talk about it soon.

:::{dropdown} ✅ Quick self-check 
**1.** Your model scores $R^2 = 0.99$ on training data but $R^2 = 0.40$ on test data. What's
wrong, and what would you change?

**2.** You fit a straight line and the residual plot shows a clear curve. Bias or variance
problem? What's the fix?

**3.** Would you rather report MAE or RMSE if a few large prediction errors are especially
costly, and why?

---
:::

:::{dropdown} ✅ Answers (try before peeking)
*Answers:*

**1.** Big train–test gap → **overfitting (high variance)**. Reduce complexity, add
regularisation, get more data, or use cross-validation to pick a simpler model.

**2.** A patterned residual = the model is **missing structure** → **high bias / underfitting**.
Fix by adding flexibility (a higher-degree term or a more expressive model).

**3.** **RMSE** - it squares errors, so it punishes big misses harder, matching the fact that
big misses cost you more.
:::

